In [1]:
!pip install langchain langgraph langchain-groq selenium beautifulsoup4 langchain-community langchain-huggingface chromadb sentence-transformers

  Using cached langchain_groq-1.1.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached langchain_groq-1.1.2-py3-none-any.whl (19 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   --- --------------------------------

In [ ]:
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "api_key"

In [2]:
from pathlib import Path
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from selenium import webdriver
from bs4 import BeautifulSoup

c:\Users\light\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def scrape_website(url):
    driver = webdriver.Chrome()
    driver.get(url)
    html = driver.page_source
    driver.quit()
    
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text[:200]

In [4]:
# scrape_website("https://beautiful-soup-4.readthedocs.io/en/latest/#quick-start")

In [5]:
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)
def explain_content(text):
    prompt = f"""
    Explain the following website content in simple terms:

    {text}
    """
    return llm.invoke(prompt).content

In [6]:
VECTOR_DB_DIR = (Path.cwd() / "vector_db")
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(
    collection_name="rag_docs",
    embedding_function=embedding_model,
    persist_directory=str(VECTOR_DB_DIR),
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Loaded RAG vector store from {VECTOR_DB_DIR}")

@tool
def scrape(url):
    """Scrape a webpage and return its content."""
    return scrape_website(url)

@tool
def explain(text):
    """Explain given text content."""
    return explain_content(text)

@tool
def rag(query: str):
    """Answer a question using the local RAG vector store."""
    matches = vectorstore.similarity_search_with_score(query, k=3)
    if not matches:
        return "No matching context was found in the RAG store."

    context_lines = []
    for rank, (doc, score) in enumerate(matches, start=1):
        context_lines.append(
            f"Source {rank} | Score: {score:.4f}\n{doc.page_content}"
        )

    context_text = "\n\n".join(context_lines)
    prompt = f"""
    You are a RAG assistant.

    Use only the retrieved context to answer the user's question.

    Question:
    {query}

    Retrieved context:
    {context_text}
    """
    return llm.invoke(prompt).content

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5527.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\light\AppData\Local\Temp\ipykernel_9092\1096236116.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Loaded RAG vector store from c:\Users\light\OneDrive\Desktop\intern\AI_agent\vector_db


In [7]:
class AgentState(TypedDict):
    tools: str
    inputs: str
    last_task:str
    web_url: str
    scraped_content: str
    last_response: str
    task: str
    history: list
    memory_window: int 

In [8]:
TOOLS = """
You have access to the following tools:

1. scrape(url: str)
   - Use this to extract content from a website URL.

2. explain(text: str)
   - Use this to explain or summarize given content.

3. rag(query: str)
   - Use this to answer questions from the local vector database.

Decide which tool to use based on the user input.
If no tool is needed, return: finish
"""

In [9]:
def scrape_node(state: AgentState):
    content = scrape.invoke({"url": state["web_url"]})
    return {
        "scraped_content": content,
        "last_response": content
    }

In [10]:
def explain_node(state: AgentState):
    explanation = explain.invoke({"text": state["scraped_content"]})
    print(explanation)
    return {
        "last_response": explanation
    }

In [11]:
def rag_node(state: AgentState):
    answer = rag.invoke({"query": state["inputs"]})
    print(answer)
    return {
        "last_response": answer
    }

In [12]:
def brain(state: AgentState):
    formatted_history = []
    for turn in state['history']:
        formatted_history.append(
            f"User: {turn.get('user', '')} | Task: {turn.get('task', '')} | Agent: {turn.get('response', '')}"
        )
    formatted_history_text = "\n".join(formatted_history)
    prompt = f"""
You are an intelligent agent.

{state['tools']}

Conversation memory (last {state['memory_window']} turns):
{formatted_history_text}

User input:
{state['inputs']}

Current state:
- scraped_content exists: {bool(state.get('scraped_content'))}

last tasks was :{state['last_task']}

Rules:
- If user asks to scrape -> use scrape
- If user asks to explain AND content exists -> use explain
- If user asks about a previous answer, earlier response, or something from history -> use recall
- If user asks to answer from the local docs, vector database, or indexed knowledge base -> use rag
- The last task and the current task should not be same
- Otherwise -> finish

Return ONLY:
scrape / explain / recall / rag / finish
"""

    decision = llm.invoke(prompt).content.strip().lower()
    return {"task": decision}

In [13]:
def recall_node(state: AgentState):
    formatted_history = []
    for turn in state['history']:
        formatted_history.append(
            f"User: {turn.get('user', '')} | Task: {turn.get('task', '')} | Agent: {turn.get('response', '')}"
        )
    formatted_history_text = "\n".join(formatted_history)

    prompt = f"""
You answer questions using the conversation memory below.

Conversation memory:
{formatted_history_text}

Current user question:
{state['inputs']}

Answer only from the stored memory. If the answer is not present, say you do not have that earlier response in memory.
"""

    answer = llm.invoke(prompt).content.strip()
    print(answer)
    return {
        "last_response": answer
    }

In [14]:
def route_task(state: AgentState):
    print(state["task"], " called \n")
    return state["task"]

In [15]:
builder = StateGraph(AgentState)

builder.add_node("brain", brain)
builder.add_node("scrape", scrape_node)
builder.add_node("explain", explain_node)
builder.add_node("rag", rag_node)
builder.add_node("recall", recall_node)

builder.set_entry_point("brain")

builder.add_conditional_edges(
    "brain",
    route_task,
    {
        "scrape": "scrape",
        "explain": "explain",
        "rag": "rag",
        "recall": "recall",
        "finish": END,
    },
)

builder.add_edge("scrape", "brain")   
builder.add_edge("explain", END)
builder.add_edge("rag", END)
builder.add_edge("recall", END)

graph = builder.compile()

In [16]:
state = {
    "inputs": "",
    "tools": TOOLS,
    "web_url": "https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant",
    "scraped_content": "",
    "last_response": "",
    "task": "",
    "history": [],
    "last_task": "",
    "memory_window": 5
}

def run_agent(user_text, web_url = None):
    global state

    state["inputs"] = user_text
    if web_url is not None:
        state["web_url"] = web_url

    result = graph.invoke(state)
    state.update(result)

    executed_task = state.get("task", "finish")
    state["last_task"] = executed_task
    state["history"].append({
        "user": user_text,
        "task": executed_task,
        "response": state.get("last_response", ""),
    })

    window = state.get("memory_window", 5)
    state["history"] = state["history"][-window:]

    return state

In [17]:
run_agent("scrape this website and explain it too","https://en.wikipedia.org/wiki/Main_Page" )


scrape  called 

scrape  called 

scrape  called 

explain  called 

Here's a simple explanation of the website content you provided:

**Wikipedia** is a free online encyclopedia (a collection of knowledge about everything!) that anyone can edit. The homepage you see has a few key parts:

1. **Main Menu**:  
   - **Main page**: The starting point of Wikipedia.  
   - **Contents**: A list of all the topics covered on Wikipedia.  
   - **Current events**: Recent news or updates.  
   - **Random article**: Takes you to a random topic to explore.  
   - **About Wikipedia**: Explains what Wikipedia is and how it works.  
   - **Contact us**: How to reach the Wikipedia team.  
   - **Contribute**: A way for people to add or improve articles.  
   - **Help**: Tips for using the website.  
   - **Learn**: Resources for learning new things.  

2. **Navigation**:  
   - You can move the sidebar (a menu on the side) or hide it to make the page cleaner.  

**Key ideas**:  
- Wikipedia is free, ope

{'inputs': 'scrape this website and explain it too',
 'tools': '\nYou have access to the following tools:\n\n1. scrape(url: str)\n   - Use this to extract content from a website URL.\n\n2. explain(text: str)\n   - Use this to explain or summarize given content.\n\n3. rag(query: str)\n   - Use this to answer questions from the local vector database.\n\nDecide which tool to use based on the user input.\nIf no tool is needed, return: finish\n',
 'web_url': 'https://en.wikipedia.org/wiki/Main_Page',
 'scraped_content': 'Wikipedia, the free encyclopedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn t',
 'last_response': "Here's a simple explanation of the website content you provided:\n\n**Wikipedia** is a free online encyclopedia (a collection of knowledge about everything!) that anyone can edit. The homepage you see has a few key parts:\n\n1. **Main Menu**:  \n   - **Mai

In [18]:
run_agent("explain this website")

finish  called 



{'inputs': 'explain this website',
 'tools': '\nYou have access to the following tools:\n\n1. scrape(url: str)\n   - Use this to extract content from a website URL.\n\n2. explain(text: str)\n   - Use this to explain or summarize given content.\n\n3. rag(query: str)\n   - Use this to answer questions from the local vector database.\n\nDecide which tool to use based on the user input.\nIf no tool is needed, return: finish\n',
 'web_url': 'https://en.wikipedia.org/wiki/Main_Page',
 'scraped_content': 'Wikipedia, the free encyclopedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn t',
 'last_response': "Here's a simple explanation of the website content you provided:\n\n**Wikipedia** is a free online encyclopedia (a collection of knowledge about everything!) that anyone can edit. The homepage you see has a few key parts:\n\n1. **Main Menu**:  \n   - **Main page**: The star

In [19]:
run_agent("scrape this website and explain it too","https://en.wikipedia.org/wiki/Talk:Main_Page" )

explain  called 

Here’s a simple explanation of the website content you provided:

---

**Wikipedia** is a free online encyclopedia (a place where people can find information on almost any topic). The homepage you see is the starting point for exploring it. Here’s what the main parts mean:

1. **"Jump to content"**: A shortcut for people using screen readers (tools for visually impaired users) to skip to the main part of the page.

2. **Main menu**: This is the top navigation bar with links to key sections:
   - **Main page**: The Wikipedia homepage (where you are now).
   - **Contents**: A list of all the topics covered on Wikipedia.
   - **Current events**: News about recent happenings.
   - **Random article**: Takes you to a randomly chosen article.
   - **About Wikipedia**: Explains what Wikipedia is and how it works.
   - **Contact us**: How to reach the Wikipedia team.
   - **Contribute**: A way for users to help edit or add new articles.
   - **Help**: Guides on how to use Wiki

{'inputs': 'scrape this website and explain it too',
 'tools': '\nYou have access to the following tools:\n\n1. scrape(url: str)\n   - Use this to extract content from a website URL.\n\n2. explain(text: str)\n   - Use this to explain or summarize given content.\n\n3. rag(query: str)\n   - Use this to answer questions from the local vector database.\n\nDecide which tool to use based on the user input.\nIf no tool is needed, return: finish\n',
 'web_url': 'https://en.wikipedia.org/wiki/Talk:Main_Page',
 'scraped_content': 'Wikipedia, the free encyclopedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn t',
 'last_response': 'Here’s a simple explanation of the website content you provided:\n\n---\n\n**Wikipedia** is a free online encyclopedia (a place where people can find information on almost any topic). The homepage you see is the starting point for exploring it. Here’s

In [20]:
run_agent("now explain both 1 vs 2 url like what was the difference of both website")

finish  called 



{'inputs': 'now explain both 1 vs 2 url like what was the difference of both website',
 'tools': '\nYou have access to the following tools:\n\n1. scrape(url: str)\n   - Use this to extract content from a website URL.\n\n2. explain(text: str)\n   - Use this to explain or summarize given content.\n\n3. rag(query: str)\n   - Use this to answer questions from the local vector database.\n\nDecide which tool to use based on the user input.\nIf no tool is needed, return: finish\n',
 'web_url': 'https://en.wikipedia.org/wiki/Talk:Main_Page',
 'scraped_content': 'Wikipedia, the free encyclopedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn t',
 'last_response': 'Here’s a simple explanation of the website content you provided:\n\n---\n\n**Wikipedia** is a free online encyclopedia (a place where people can find information on almost any topic). The homepage you see is the start